# RAG Layer — Vector Search Index Build

Build order step 6 (`CLAUDE.md`): real semantic search to replace the keyword-match proxies
that `search_reviews()` (Sentiment) and `search_policy_docs()` (Compliance) shipped with
initially. This notebook builds both Vector Search collections and demonstrates them working.

## The architecture decision: local embeddings, not Databricks Vector Search

CLAUDE.md's stack table names Databricks Vector Search specifically. That's a real, separately
billed managed service — its own compute endpoint on top of what we're already running. Given
the same cost-consciousness that drove the Groq decision for the boss agent, this RAG layer
uses **local, free embeddings** instead:

| | Databricks Vector Search | Local embeddings (chosen) |
|---|---|---|
| Cost | Separate billed endpoint | $0 — runs on CPU |
| Setup | Provision an endpoint, sync index | `pip install sentence-transformers` |
| Latency | Network round-trip | In-process, milliseconds |
| Scale ceiling | Production, millions of vectors | Fine into the low millions; this dataset has ~41k reviews + a handful of doc sections |

This is the same kind of pragmatic substitution as Groq for the boss LLM: matches the
*capability* CLAUDE.md calls for, not the exact vendor name, with the door open to swap in real
Databricks Vector Search later if this ever needs to scale past what a single machine can hold
in memory.

## The embedding model: multilingual, no translation step

Olist reviews are mostly Portuguese (flagged back in CLAUDE.md section 2). Two ways to handle
that: translate-then-embed-with-English-model, or use a model that natively embeds both
languages into the same vector space. This uses the second approach —
**paraphrase-multilingual-MiniLM-L12-v2** — which means a query typed in English still
retrieves relevant Portuguese reviews correctly, with one fewer moving part (no translation
step to get wrong).

## Two separate collections, per CLAUDE.md section 2

Reviews and institutional documents get their own `VectorIndex` instances
(`backend/rag/review_index.py`, `backend/rag/policy_index.py`) — different metadata shapes
(`review_id`/`order_id`/`review_score` vs. `document`/`section`), never mixed into one
collection. The policy documents are chunked by section header (clause-addressable, matching
CLAUDE.md's "smaller chunks, section/clause-based chunking" guidance for these docs), reusing
the same parser the Compliance agent already had (`document_loader.py`).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.db import DatabricksClient
from backend.rag import build_policy_index, build_review_index, get_policy_index, get_review_index

db = DatabricksClient()

## Build (or skip if already built)

Idempotent — re-running this notebook doesn't re-embed ~41k reviews every time. Set `force_rebuild = True` to rebuild after the underlying data or documents change.

In [2]:
force_rebuild = False

def _is_built(get_fn) -> bool:
    try:
        get_fn()
        return True
    except RuntimeError:
        return False

if force_rebuild or not _is_built(get_policy_index):
    print("Building policy index...")
    build_policy_index()
    print("done")
else:
    print("Policy index already built, skipping (set force_rebuild=True to rebuild)")

if force_rebuild or not _is_built(get_review_index):
    print("Building review index (~41k reviews, takes a few minutes on CPU)...")
    build_review_index(db)
    print("done")
else:
    print("Review index already built, skipping (set force_rebuild=True to rebuild)")

Policy index already built, skipping (set force_rebuild=True to rebuild)
Review index already built, skipping (set force_rebuild=True to rebuild)


## Test: policy document search

A query about delivery requirements should surface the SLA clause specifically, ranked above less-relevant sections.

In [3]:
policy_index = get_policy_index()
for r in policy_index.search("what is the on-time delivery requirement for sellers", top_k=3):
    print(f"[{r['score']}] {r['doc_type']} / {r['section']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[0.6156] contract / Service Level Agreement (SLA)
[0.5711] contract / Payment Terms
[0.4798] contract / Scope


## Test: cross-lingual review search

The real test of the multilingual embedding decision — an **English** query should still retrieve relevant **Portuguese** reviews, with no translation step anywhere in the pipeline.

In [4]:
review_index = get_review_index()

print("English query: 'product arrived broken'")
for r in review_index.search("product arrived broken", top_k=5):
    print(f"  [{r['score']}] score={r['review_score']} \"{r['text'][:70]}\"")

print("\nPortuguese query: 'entrega atrasada' (delayed delivery)")
for r in review_index.search("entrega atrasada", top_k=5):
    print(f"  [{r['score']}] score={r['review_score']} \"{r['text'][:70]}\"")

English query: 'product arrived broken'
  [0.8606] score=1 "O produto chegou quebrado"
  [0.8468] score=1 "O produto chegou quebrado."
  [0.8463] score=1 "O produto está quebrado"
  [0.8438] score=1 "o produto veio quebrado"
  [0.8397] score=2 "O produto chegou rasgado "

Portuguese query: 'entrega atrasada' (delayed delivery)
  [0.9655] score=5 "A entrega atrasou"
  [0.91] score=1 "A entrega esta atrasada."
  [0.8783] score=1 "demora na entrega"
  [0.8709] score=1 "entrega muito atrasada."
  [0.8629] score=1 "Sem satisfação sobre atraso na entrega"


## Verify the upgrade through the actual agent tools

`search_reviews()` and `search_policy_docs()` now call into this RAG layer instead of the
SQL `LIKE` / keyword-match proxies they shipped with.

In [5]:
from backend.agents.sentiment import SentimentAgent
from backend.agents.compliance import ComplianceAgent

sentiment_agent = SentimentAgent(db)
briefing = sentiment_agent.run("Are customers unhappy about broken or damaged products?")
search_finding = [f for f in briefing.findings if f.source == "search_reviews"][0]
print(f"[{search_finding.severity.upper()}] (confidence {search_finding.confidence:.2f}) {search_finding.claim}")

compliance_agent = ComplianceAgent(db)
briefing = compliance_agent.run("What are our SLA obligations to sellers?")
search_finding = [f for f in briefing.findings if f.source == "search_policy_docs"][0]
print(f"[{search_finding.severity.upper()}] (confidence {search_finding.confidence:.2f}) {search_finding.claim}")

[INFO] (confidence 0.70) Semantic search for 'Are customers unhappy about broken or damaged products?' matched 10 reviews above the relevance threshold


[INFO] (confidence 0.70) Semantic search for 'What are our SLA obligations to sellers?' matched 4 document sections above the relevance threshold


## Through the boss agent

A query aimed squarely at product-quality complaints should now retrieve real matching
reviews (previously: zero matches for any query not phrased in exact Portuguese keywords).

In [6]:
from backend.agents.boss import BossAgent

boss = BossAgent(db)
rec = boss.run("What are customers complaining about with damaged or broken products?")

print(f"Agents invoked: {[a.value for a in rec.agents_invoked]}")
print(f"Overall confidence: {rec.confidence_overall}")
print(f"\nSynthesis:\n{rec.synthesis}")

Agents invoked: ['Sentiment']
Overall confidence: 0.71

Synthesis:
**Board Memo – Customer Complaints on Damaged or Broken Products**

*Sentiment Agent (Sentiment tool)*:
- Semantic search identified 10 highly relevant reviews discussing damaged/broken items (confidence 0.7).
- The lowest‑rated product (ID 0c40401eba358c9ef2ff2df80b6eab52) received a 1.0 / 5 rating from 5 reviews, indicating a critical quality issue (confidence 0.85).
- Negative‑review share (score ≤ 2) has remained flat over the past 12 months, suggesting a persistent problem rather than a recent spike (confidence 0.85).
- Top recurring terms in negative reviews (Portuguese) are *produto*, *recebi*, *comprei*, *veio*, *entrega*, pointing to delivery‑related damage or product defects (confidence 0.5).
- Sentiment varies by region, ranging from 4.19 / 5 in AP to 3.61 / 5 in RR, indicating regional disparities in product quality or delivery experience (confidence 0.8).
- Correlation between product listing photo count an

## What's still pending

Per the build order in `CLAUDE.md`:

7. Governance logging middleware persistence + MongoDB
8. MERN frontend + streaming + WebSocket agent status
9. Eval suite
10. Deployment containerization